In [36]:
import sqlite3
import pandas as pd
import time
import shutil

# Create a copy of the database to work on
source_db = "../Datasets/database/movies.db"
target_db = "Outputs/movies.db"
shutil.copy(source_db, target_db)

# Connect to SQLite database
conn = sqlite3.connect(target_db)
cursor = conn.cursor()


In [37]:
# Function to execute SQL queries and measure execution time
def execute_query(query, fetch=True):
    start_time = time.time()
    cursor.execute(query)
    result = cursor.fetchall() if fetch else None
    elapsed_time = time.time() - start_time
    print(f"Query executed in {elapsed_time:.5f} seconds")
    return result

In [ ]:
# Display schema
print("Schema of movies.db:")
schema = execute_query("SELECT sql FROM sqlite_master;")
for row in schema:
    print(row[0])

# # Peek at movies table
# print("\nSample records from movies table:")
# df_movies = pd.read_sql_query("SELECT * FROM movies LIMIT 5;", conn)
# # print(df_movies)

### Primary key indexing

In [16]:
# Search for a specific movie
query = "SELECT * FROM movies WHERE title = 'Inception';"
print("\nSearching for movie 'Inception' (without index):")
execute_query(query)


Searching for movie 'Inception' (without index):
Query executed in 0.00000 seconds


[(1, 'Inception', 2010)]

In [27]:
# Create index on title column
print("\nCreating index on title column...")
execute_query("CREATE INDEX IF NOT EXISTS title_index ON movies (title);", fetch=False)


Creating index on title column...
Query executed in 0.00371 seconds


In [18]:
# Verify index creation
print("\nSchema after index creation:")
schema = execute_query("SELECT sql FROM sqlite_master WHERE type='index';")
for row in schema:
    print(row[0])


Schema after index creation:
Query executed in 0.00000 seconds
None
None
CREATE INDEX title_index ON movies (title)


In [19]:
# Search again with index
print("\nSearching for movie 'Inception' (with index):")
execute_query(query)


Searching for movie 'Inception' (with index):
Query executed in 0.00000 seconds


[(1, 'Inception', 2010)]

In [20]:
# Show query execution plan
print("\nQuery Plan (with index):")
plan = execute_query("EXPLAIN QUERY PLAN " + query)
for row in plan:
    print(row)


Query Plan (with index):
Query executed in 0.00000 seconds
(3, 0, 0, 'SEARCH movies USING INDEX title_index (title=?)')


In [24]:
# Drop index
print("\nDropping index title_index...")
execute_query("DROP INDEX IF EXISTS title_index;", fetch=False)


Dropping index title_index...
Query executed in 0.00000 seconds


In [28]:
# Show query plan after index removal
print("\nQuery Plan (without index):")
plan = execute_query("EXPLAIN QUERY PLAN " + query)
for row in plan:
    print(row) #!FIX


Query Plan (without index):
Query executed in 0.00000 seconds
(3, 0, 0, 'SEARCH movies USING INDEX title_index (title=?)')


In [29]:
print("Schema of movies.db:")
schema = execute_query("SELECT sql FROM sqlite_master;")
for row in schema:
    print(row[0])

Schema of movies.db:
Query executed in 0.00000 seconds
CREATE TABLE "movies" (
    "id" INTEGER,
    "title" TEXT NOT NULL,
    "year" NUMERIC,
    PRIMARY KEY("id")
)
CREATE TABLE "people" (
    "id" INTEGER,
    "name" TEXT NOT NULL,
    "birth" NUMERIC,
    PRIMARY KEY("id")
)
CREATE TABLE "ratings" (
    "id" INTEGER,
    "movie_id" INTEGER UNIQUE,
    "rating" REAL NOT NULL,
    "votes" INTEGER NOT NULL,
    PRIMARY KEY("id"),
    FOREIGN KEY("movie_id") REFERENCES "movies"("id")
)
None
CREATE TABLE "stars" (
    "movie_id" INTEGER,
    "person_id" INTEGER,
    PRIMARY KEY("movie_id", "person_id"),
    FOREIGN KEY("movie_id") REFERENCES "movies"("id"),
    FOREIGN KEY("person_id") REFERENCES "people"("id")
)
None
CREATE INDEX title_index ON movies (title)


### Foreign Key Indexing

In [38]:
# Demonstrate foreign key indexing
query_hanks = """
SELECT title FROM movies WHERE id IN (
    SELECT movie_id FROM stars WHERE person_id = (
        SELECT id FROM people WHERE name = 'Tom Hanks'
    )
);
"""
print("\nSearching for movies starring Tom Hanks (without index):")
execute_query(query_hanks)


Searching for movies starring Tom Hanks (without index):
Query executed in 0.00100 seconds


[('Forrest Gump',)]

In [39]:
# Show query plan with indexes
print("\nQuery Plan (with no indexes):")
plan = execute_query("EXPLAIN QUERY PLAN " + query_hanks)
for row in plan:
    print(row)


Query Plan (with no indexes):
Query executed in 0.00000 seconds
(2, 0, 0, 'SEARCH movies USING INTEGER PRIMARY KEY (rowid=?)')
(6, 0, 0, 'LIST SUBQUERY 2')
(8, 6, 0, 'SCAN stars')
(13, 6, 0, 'SCALAR SUBQUERY 1')
(17, 13, 0, 'SCAN people')


In [40]:
# Create indexes on foreign keys
print("\nCreating indexes on person_id and name columns...")
execute_query("CREATE INDEX IF NOT EXISTS person_index ON stars (person_id);", fetch=False)
execute_query("CREATE INDEX IF NOT EXISTS name_index ON people (name);", fetch=False)


Creating indexes on person_id and name columns...
Query executed in 0.00400 seconds
Query executed in 0.00300 seconds


In [41]:
# Rerun query with indexes
print("\nSearching for movies starring Tom Hanks (with indexes):")
execute_query(query_hanks)


Searching for movies starring Tom Hanks (with indexes):
Query executed in 0.00100 seconds


[('Forrest Gump',)]

In [42]:
# Show query plan with indexes
print("\nQuery Plan (with indexes):")
plan = execute_query("EXPLAIN QUERY PLAN " + query_hanks)
for row in plan:
    print(row)


Query Plan (with indexes):
Query executed in 0.00000 seconds
(2, 0, 0, 'SEARCH movies USING INTEGER PRIMARY KEY (rowid=?)')
(6, 0, 0, 'LIST SUBQUERY 2')
(9, 6, 0, 'SEARCH stars USING INDEX person_index (person_id=?)')
(12, 6, 0, 'SCALAR SUBQUERY 1')
(16, 12, 0, 'SEARCH people USING COVERING INDEX name_index (name=?)')


### Demonstrate partial indexing

In [44]:
print("\nSearching for movies released in 1994 (without index):")
execute_query("SELECT title FROM movies WHERE year = 1994;")


Searching for movies released in 1994 (without index):
Query executed in 0.00000 seconds


[('Forrest Gump',), ('The Shawshank Redemption',)]

In [45]:
print("\nCreating partial index for movies in 1994...")
execute_query("CREATE INDEX IF NOT EXISTS recents ON movies (title) WHERE year = 1994;", fetch=False)


Creating partial index for movies in 1994...
Query executed in 0.00601 seconds


In [46]:
print("\nSearching for movies released in 1994 (with index):")
execute_query("SELECT title FROM movies WHERE year = 1994;")


Searching for movies released in 1994 (with index):
Query executed in 0.00000 seconds


[('Forrest Gump',), ('The Shawshank Redemption',)]

In [47]:
print("\nQuery Plan (with index):")
plan = execute_query("EXPLAIN QUERY PLAN SELECT title FROM movies WHERE year = 1994;")
for row in plan:
    print(row)


Query Plan (with index):
Query executed in 0.00000 seconds
(2, 0, 0, 'SCAN movies USING COVERING INDEX recents')


### Cleaning

In [48]:
# Drop indexes and reclaim space
print("\nDropping indexes and running VACUUM...")
execute_query("DROP INDEX IF EXISTS title_index;", fetch=False)
execute_query("DROP INDEX IF EXISTS person_index;", fetch=False)
execute_query("DROP INDEX IF EXISTS name_index;", fetch=False)
execute_query("VACUUM;", fetch=False)

# Close connection
conn.close()
print("\nDatabase connection closed.")



Dropping indexes and running VACUUM...
Query executed in 0.00100 seconds
Query executed in 0.00500 seconds
Query executed in 0.00400 seconds
Query executed in 0.00400 seconds

Database connection closed.
